In [1]:
!pip install marker-pdf -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.7/195.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 110.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.4/226.4 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796

In [10]:
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_path}")

Saving main.pdf to main.pdf
Uploaded: main.pdf


In [11]:
!pip install pytesseract pillow pymupdf -q
!apt-get install -y tesseract-ocr -q

# Verify everything is available
import importlib
for pkg in ["pytesseract", "fitz", "PIL"]:
    try:
        importlib.import_module(pkg)
        print(f"✓ {pkg}")
    except ImportError:
        print(f"✗ {pkg} — FAILED")

# Verify tesseract binary
import subprocess
result = subprocess.run(["tesseract", "--version"], capture_output=True, text=True)
if result.returncode == 0:
    print(f"✓ tesseract: {result.stdout.splitlines()[0]}")
else:
    print("✗ tesseract binary not found")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
✓ pytesseract
✓ fitz
✓ PIL
✓ tesseract: tesseract 4.1.1


In [12]:
import hashlib, json, re, io
from pathlib import Path
from dataclasses import asdict, dataclass, field
from typing import List

# ── Data Models ───────────────────────────────────────────────────────────────

@dataclass
class SlideMarkdown:
    slide_id:     str
    page_number:  int
    raw_markdown: str
    doc_id:       str

@dataclass
class SlideContent:
    slide_id:        str
    page_number:     int
    doc_id:          str
    headings:        List[str] = field(default_factory=list)
    body_text:       List[str] = field(default_factory=list)
    bullets:         List[str] = field(default_factory=list)
    table_cells:     List[str] = field(default_factory=list)
    captions:        List[str] = field(default_factory=list)
    discarded:       List[str] = field(default_factory=list)
    clean_text:      str       = ""
    heading_phrases: List[str] = field(default_factory=list)
    ocr_applied:     bool      = False

# ── Config ────────────────────────────────────────────────────────────────────

PDF_PATH = pdf_path          # from Cell 2 upload
DOC_ID   = Path(pdf_path).stem
PAGE_SEP = "-" * 48

# ── Noise patterns ────────────────────────────────────────────────────────────

NOISE_PATTERNS = [
    r"^\s*page\s+\d+\s*(of\s+\d+)?\s*$",       # page numbers
    r"^\s*\d+\s*$",                               # standalone digits
    r"^©.*",                                      # copyright
    r"^\s*(references|bibliography)\s*$",         # section headers
    r"https?://\S+",                              # URLs
    r"^\s*\[\d+\]",                               # citation markers [1]
    r"^\!\[.*?\]\(.*?\)$",                        # FIX 2: image refs ![]()
    r"^(input|output|legend|infrastructure)"      # FIX 5: diagram legend labels
    r"(\s+replaced.*|\s+kept.*|\s+storage.*)?$",
]

def is_noise(text: str) -> bool:
    t = text.strip()
    if not t:
        return True
    for pattern in NOISE_PATTERNS:
        if re.match(pattern, t, re.IGNORECASE):
            return True
    return False

# ── FIX 2: OCR noise detector ─────────────────────────────────────────────────

def is_ocr_noise(text: str) -> bool:
    """Detect garbled OCR output — not worth keeping."""
    t = text.strip()
    if not t or len(t) < 4:
        return True
    words = t.split()
    # Single garbled token that is not a real word
    if len(words) == 1 and not words[0].isalpha():
        return True
    # High ratio of non-ASCII characters
    non_ascii = sum(1 for c in t if ord(c) > 127)
    if non_ascii / len(t) > 0.1:
        return True
    # Too few actual letters vs total characters (symbols, digits dominating)
    letters = sum(1 for c in t if c.isalpha())
    if len(t) > 5 and letters / len(t) < 0.4:
        return True
    return False

# ── Step 1: Run Marker ────────────────────────────────────────────────────────

print("Loading Marker models (first run ~4 GB download, ~5 min)...")

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict

model_dict = create_model_dict()
converter  = PdfConverter(
    artifact_dict=model_dict,
    config={"paginate_output": True, "page_separator": PAGE_SEP},
)
rendered      = converter(PDF_PATH)
full_markdown = rendered.markdown
raw_pages     = [p.strip() for p in full_markdown.split(PAGE_SEP)]

print(f"Marker produced {len(raw_pages)} raw pages")

# ── Step 1b: OCR fallback for image-only pages ────────────────────────────────

print("Setting up OCR fallback...")
import subprocess
subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "-q"],
               capture_output=True)

import pytesseract
import fitz
from PIL import Image

def is_empty_page(markdown: str) -> bool:
    """True if Marker found no real text on this page."""
    cleaned   = markdown.strip()
    # Marker placeholder
    if cleaned in ["{0}", ""]:
        return True
    # Strip all image references and check what is left
    text_only = re.sub(r"!\[.*?\]\(.*?\)", "", cleaned).strip()
    # Strip all markdown symbols
    text_only = re.sub(r"[#>\-\*\|]", "", text_only).strip()
    return len(text_only) < 10

def ocr_full_page(pdf_path: str, page_number: int) -> str:
    """Render full page at 3x zoom and run Tesseract on it."""
    doc  = fitz.open(pdf_path)
    page = doc[page_number - 1]       # zero-indexed
    mat  = fitz.Matrix(3, 3)          # 3x zoom for better accuracy
    pix  = page.get_pixmap(matrix=mat)
    img  = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    text = pytesseract.image_to_string(img, config="--psm 6")
    doc.close()
    return text.strip()

slides_md = []
for idx, page_md in enumerate(raw_pages, start=1):
    page_md = page_md.strip()

    if is_empty_page(page_md):
        print(f"  Slide {idx:03d}: Marker found no text → running OCR...")
        ocr_text = ocr_full_page(PDF_PATH, idx)
        if ocr_text:
            # Wrap each non-empty line as OCR blockquote
            page_md = "\n".join(
                f"> OCR: {line}"
                for line in ocr_text.splitlines()
                if line.strip()
            )
            print(f"  Slide {idx:03d}: OCR recovered {len(ocr_text)} chars")
        else:
            print(f"  Slide {idx:03d}: OCR found nothing — slide may be decorative")
            page_md = ""

    if not page_md:
        continue

    slides_md.append(SlideMarkdown(
        slide_id    = f"slide_{idx:03d}",
        page_number = idx,
        raw_markdown= page_md,
        doc_id      = DOC_ID,
    ))

print(f"\nTotal slides after OCR fallback: {len(slides_md)}")

# ── Step 2: Preprocess each slide ─────────────────────────────────────────────

# FIX 3: OCR lines re-routed through normal classification
def classify_ocr_line(text: str, content: SlideContent) -> None:
    """
    Parse a single OCR-recovered line and place it in the
    correct bucket instead of forcing everything into captions.
    """
    stripped = text.strip()
    if not stripped or is_noise(stripped) or is_ocr_noise(stripped):
        if stripped:
            content.discarded.append(stripped)
        return

    # Heading heuristic: ALL CAPS, or ends with known structural keywords
    # or explicitly starts with # (rare in OCR but possible)
    if stripped.startswith("#"):
        inner = re.sub(r"^#+\s*", "", stripped).strip()
        if inner:
            content.headings.append(inner)
            content.heading_phrases.append(inner)

    elif stripped.startswith(("- ", "* ", "+ ")):
        inner = stripped[2:].strip()
        if inner:
            content.bullets.append(inner)

    elif stripped.startswith("|") and stripped.endswith("|"):
        if not re.match(r"^\|[\s\-\|]+\|$", stripped):
            cells = [c.strip() for c in stripped.split("|") if c.strip()]
            content.table_cells.extend(cells)

    else:
        # Default: body text (not caption — no weight penalty)
        content.body_text.append(stripped)


def parse_markdown_to_content(slide_md: SlideMarkdown) -> SlideContent:
    content = SlideContent(
        slide_id    = slide_md.slide_id,
        page_number = slide_md.page_number,
        doc_id      = slide_md.doc_id,
    )
    lines = slide_md.raw_markdown.split("\n")

    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue

        # ── Headings ──────────────────────────────────────────────
        if stripped.startswith("#"):
            text = re.sub(r"^#+\s*", "", stripped).strip()
            if text and not is_noise(text):
                content.headings.append(text)
                content.heading_phrases.append(text)
            elif text:
                content.discarded.append(text)

        # ── Bullets ───────────────────────────────────────────────
        elif stripped.startswith(("- ", "* ", "+ ")):
            text = stripped[2:].strip()
            if text and not is_noise(text):
                content.bullets.append(text)
            elif text:
                content.discarded.append(text)

        # ── Table rows ────────────────────────────────────────────
        elif stripped.startswith("|") and stripped.endswith("|"):
            if re.match(r"^\|[\s\-\|]+\|$", stripped):
                continue
            cells = [c.strip() for c in stripped.split("|") if c.strip()]
            for cell in cells:
                if cell and not is_noise(cell):
                    content.table_cells.append(cell)

        # ── Blockquotes: real captions OR OCR-recovered lines ─────
        elif stripped.startswith(">"):
            text = stripped.lstrip(">").strip()

            if text.startswith("OCR:"):
                # FIX 3: strip prefix, re-classify properly
                inner = text[4:].strip()
                content.ocr_applied = True
                classify_ocr_line(inner, content)
            else:
                # Real Marker blockquote/caption
                if text and not is_noise(text):
                    content.captions.append(text)
                elif text:
                    content.discarded.append(text)

        # ── Body text ─────────────────────────────────────────────
        else:
            if not is_noise(stripped):
                content.body_text.append(stripped)
            else:
                content.discarded.append(stripped)

    return content


slide_contents = [parse_markdown_to_content(s) for s in slides_md]

# ── Stage 3: Cross-slide repetition filter ────────────────────────────────────

from collections import Counter

def cross_slide_filter(slide_contents: List[SlideContent]) -> List[SlideContent]:
    total = len(slide_contents)
    if total == 0:
        return slide_contents

    # FIX 1: skip filter entirely for small documents
    MIN_SLIDES_FOR_FILTER = 5
    if total < MIN_SLIDES_FOR_FILTER:
        print(f"Cross-slide filter skipped "
              f"(only {total} slides — need {MIN_SLIDES_FOR_FILTER}+)")
        return slide_contents

    all_blocks = []
    for sc in slide_contents:
        all_blocks.extend(
            sc.headings + sc.body_text +
            sc.bullets  + sc.table_cells + sc.captions
        )

    counts  = Counter(all_blocks)
    repeats = {t for t, c in counts.items() if c / total > 0.5}

    if repeats:
        print(f"Cross-slide filter removed {len(repeats)} repeated blocks:")
        for r in repeats:
            print(f"  '{r[:70]}'")
    else:
        print("Cross-slide filter: nothing removed")

    def remove_repeats(lst):
        kept      = [t for t in lst if t not in repeats]
        discarded = [t for t in lst if t     in repeats]
        return kept, discarded

    for sc in slide_contents:
        sc.headings,    d1 = remove_repeats(sc.headings)
        sc.body_text,   d2 = remove_repeats(sc.body_text)
        sc.bullets,     d3 = remove_repeats(sc.bullets)
        sc.table_cells, d4 = remove_repeats(sc.table_cells)
        sc.captions,    d5 = remove_repeats(sc.captions)
        sc.discarded.extend(d1 + d2 + d3 + d4 + d5)

    return slide_contents


slide_contents = cross_slide_filter(slide_contents)

# ── Stage 4: Assemble clean_text ──────────────────────────────────────────────

for sc in slide_contents:
    sc.clean_text = " ".join(
        sc.headings    +
        sc.body_text   +
        sc.bullets     +
        sc.table_cells +
        sc.captions
    ).strip()

# ── Save both outputs ─────────────────────────────────────────────────────────

stem      = Path(PDF_PATH).stem
raw_out   = f"{stem}_parsed.json"
clean_out = f"{stem}_preprocessed.json"

with open(raw_out, "w", encoding="utf-8") as f:
    json.dump([asdict(s) for s in slides_md],
              f, ensure_ascii=False, indent=2)

with open(clean_out, "w", encoding="utf-8") as f:
    json.dump([asdict(s) for s in slide_contents],
              f, ensure_ascii=False, indent=2)

# ── Summary ───────────────────────────────────────────────────────────────────

print(f"\n{'='*50}")
print(f"DONE")
print(f"{'='*50}")
print(f"Total slides         : {len(slide_contents)}")
print(f"Slides with OCR      : {sum(1 for s in slide_contents if s.ocr_applied)}")
print(f"{'='*50}")
for sc in slide_contents:
    print(f"\n[{sc.slide_id}]")
    print(f"  headings    : {len(sc.headings)}")
    print(f"  body_text   : {len(sc.body_text)}")
    print(f"  bullets     : {len(sc.bullets)}")
    print(f"  table_cells : {len(sc.table_cells)}")
    print(f"  captions    : {len(sc.captions)}")
    print(f"  discarded   : {len(sc.discarded)}")
    print(f"  ocr_applied : {sc.ocr_applied}")
print(f"\nRaw JSON      → {raw_out}")
print(f"Preprocessed  → {clean_out}")

Loading Marker models (first run ~4 GB download, ~5 min)...


Running OCR Error Detection: 100%|██████████| 1/1 [00:00<00:00, 76.67it/s]
Detecting bboxes: 0it [00:00, ?it/s]
Detecting bboxes: 0it [00:00, ?it/s]


Marker produced 2 raw pages
Setting up OCR fallback...
  Slide 001: Marker found no text → running OCR...
  Slide 001: OCR recovered 2445 chars

Total slides after OCR fallback: 2
Cross-slide filter skipped (only 2 slides — need 5+)

DONE
Total slides         : 2
Slides with OCR      : 1

[slide_001]
  headings    : 0
  body_text   : 58
  bullets     : 0
  table_cells : 0
  captions    : 0
  discarded   : 2
  ocr_applied : True

[slide_002]
  headings    : 1
  body_text   : 1
  bullets     : 0
  table_cells : 0
  captions    : 0
  discarded   : 2
  ocr_applied : False

Raw JSON      → main_parsed.json
Preprocessed  → main_preprocessed.json


In [13]:
for sc in slide_contents[:3]:
    print(f"\n{'='*60}")
    print(f"[{sc.slide_id}]  page {sc.page_number}")
    print(f"Headings    : {sc.headings}")
    print(f"Bullets     : {sc.bullets[:3]}")
    print(f"Body        : {sc.body_text[:2]}")
    print(f"Table cells : {sc.table_cells[:3]}")
    print(f"Captions    : {sc.captions[:2]}")
    print(f"OCR applied : {sc.ocr_applied}")
    print(f"\nclean_text preview:\n{sc.clean_text[:300]}")


[slide_001]  page 1
Headings    : []
Bullets     : []
Body        : ['PACE-KG: Optimized EduKG Construction Pipeline', 'Pedagogically-Aware, Citation-Evidenced Knowledge Graph']
Table cells : []
Captions    : []
OCR applied : True

clean_text preview:
PACE-KG: Optimized EduKG Construction Pipeline Pedagogically-Aware, Citation-Evidenced Knowledge Graph PDF Learning Material Raw PDF pages / slides Layer 1 Handles text - tables - figures Marker Parsing equations - layout detection Multimodal - Layout-aware Replaces PDFMiner + all patches Raw markdo

[slide_002]  page 2
Headings    : ['PACE-KG: Optimized EduKG Construction Pipeline']
Bullets     : []
Body        : ['Pedagogically-Aware, Citation-Evidenced Knowledge Graph']
Table cells : []
Captions    : []
OCR applied : False

clean_text preview:
PACE-KG: Optimized EduKG Construction Pipeline Pedagogically-Aware, Citation-Evidenced Knowledge Graph


In [14]:
from IPython.display import Markdown, display

# ── View full raw markdown per slide ─────────────────────────────────────────

# Change these to look at specific slides
SHOW_SLIDES = [1, 2, 3]   # page numbers to preview
SHOW_FULL   = False        # True = full markdown, False = first 500 chars

for slide in slides_md:
    if slide.page_number not in SHOW_SLIDES:
        continue

    print(f"\n{'='*60}")
    print(f"SLIDE {slide.page_number}  ({slide.slide_id})")
    print(f"{'='*60}")

    # Raw markdown text
    md = slide.raw_markdown
    preview = md if SHOW_FULL else md[:500] + ("..." if len(md) > 500 else "")
    print(preview)


SLIDE 1  (slide_001)
> OCR: PACE-KG: Optimized EduKG Construction Pipeline
> OCR: Pedagogically-Aware, Citation-Evidenced Knowledge Graph
> OCR: PDF Learning
> OCR: Material
> OCR: Raw PDF pages / slides
> OCR: Layer 1 Handles text - tables - figures
> OCR: Marker Parsing equations - layout detection
> OCR: Multimodal - Layout-aware Replaces PDFMiner + all patches
> OCR: Raw markdown per slide
> OCR: Parse markdown tags into types
> OCR: Layer 2 Discard: footers - page numbers
> OCR: Markdown Preprocessor references...

SLIDE 2  (slide_002)
## PACE-KG: Optimized EduKG Construction Pipeline

Pedagogically-Aware, Citation-Evidenced Knowledge Graph

![](_page_0_Figure_2.jpeg)

Input Replaced/Novel Kept / Improved Storage / Output Infrastructure Legend


In [15]:
# Renders markdown properly in Colab output
# Images won't show but headings, tables, bullets will render nicely

for slide in slides_md:
    if slide.page_number not in SHOW_SLIDES:
        continue

    display(Markdown(f"---\n### Slide {slide.page_number} — `{slide.slide_id}`\n---"))
    display(Markdown(slide.raw_markdown))

---
### Slide 1 — `slide_001`
---

> OCR: PACE-KG: Optimized EduKG Construction Pipeline
> OCR: Pedagogically-Aware, Citation-Evidenced Knowledge Graph
> OCR: PDF Learning
> OCR: Material
> OCR: Raw PDF pages / slides
> OCR: Layer 1 Handles text - tables - figures
> OCR: Marker Parsing equations - layout detection
> OCR: Multimodal - Layout-aware Replaces PDFMiner + all patches
> OCR: Raw markdown per slide
> OCR: Parse markdown tags into types
> OCR: Layer 2 Discard: footers - page numbers
> OCR: Markdown Preprocessor references - repeated headers
> OCR: Noi MeRGontantitect Cross-slide repetition filter
> OCR: o1se remova! - Vontent typing Heading-emphasis preservation
> OCR: Clean typed text (headings - body - bullets - tables)
> OCR: Adaptive cap (up to 30)
> OCR: Layer 3 Quality threshold > 0.3
> OCR: Keyphrase Extraction Neer: mst contain poun
> OCR: . oun-chunk cross-validation
> OCR: SIFRankSqueezeBERT + adaptive filter Heading boost +0.2
> OCR: Scored keyphrases (10-25) + slide text
> OCR: Keyphrases as hard anchors
> OCR: Evidence citation required
> OCR: Models Relation schema:
> OCR: SBERT: all-mpnet-base-v2 Layer 4 [NOVEL] isPrerequisiteOf » isDefinedAs
> OCR: LLM: GPT-4o-mini (primary) LLM Triple Extraction isExampleOf - contrasted With
> OCR: Llama 3.1 8B (secondary) . . appliedIn - isPartOf
> OCR: spaCy: en_core_web.sm Coe eeu c eg ae CR cel ate causeOf - isGeneralizationOf
> OCR: 3-layer hallucination filter
> OCR: Replaces DBpedia Spotlight
> OCR: {subject, relation, object, evidence, confidence}
> OCR: L w = 0.5 Wevid + 0.3 Wslide + 0.2 Waoe
> OCR: ayer 5 + relation-role boost
> OCR: Concept Weighting & Pruning + source-type boost (heading)
> OCR: . . rune below thresholc
> OCR: Three-signal SBERT - Relation boost No Wikipedia needed
> OCR: Weighted MCs - low-weight triples pruned
> OCR: Document Vocabulary Phase 1: build doc vocabulary
> OCR: All surviving concepts Layer 6 [NOVEL] Phase 2: LLM selects from pool only
> OCR: across all slides Closed-Corpus Expansion Phase 3: SBERT similarity gate
> OCR: Marker noun chunks D A ioaiil rn Phase 4: slide-scope constraint
> OCR: Stored in memory / Redis ee ee Miele Eos nes eRe necead Replaces Wikipedia dump entirely
> OCR: Available to
> OCR: learners MCs + related concepts (doc-bounded)
> OCR: per slide
> OCR: Redis + Celery
> OCR: aera ern te Layer 7
> OCR: Horizontal scalability Slide-EduKG (Neo4j)
> OCR: Auto-requeue on failure A . . .
> OCR: neon ern arn cee Stored per slide - Available immediately
> OCR: Aggregated after all slides processed
> OCR: Nodes: concepts + aliases + weight
> OCR: Layer 8 Edges: typed pedagogical relations
> OCR: LM-EduKG pevery edge: evidence citation |
> OCR: . rovenance: slide + sentence
> OCR: Caen ae Ce Siena Expansion edges flagged separately
> OCR: Legend
> OCR: Storage
> OCR: Input Replaced/Novel Kept / Improved Out & if Infrastructure
> OCR: utpu

---
### Slide 2 — `slide_002`
---

## PACE-KG: Optimized EduKG Construction Pipeline

Pedagogically-Aware, Citation-Evidenced Knowledge Graph

![](_page_0_Figure_2.jpeg)

Input Replaced/Novel Kept / Improved Storage / Output Infrastructure Legend

In [16]:
from google.colab import files
files.download(raw_out)
files.download(clean_out)
print("Check your browser Downloads folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Check your browser Downloads folder.
